## Import & Path

In [17]:
from pathlib import Path
import json
import re

import pandas as pd


ROOT = Path("..")

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = ROOT / "outputs"
MODEL_DIR = ROOT / "models"

TRAIN_PATH = PROCESSED_DIR / "train.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"

KG_PATH = OUTPUT_DIR / "kg_train.tsv"
KG_RELATIONS_PATH = OUTPUT_DIR / "kg_relations.txt"

TRAIN_FEATURES_PATH = (
    OUTPUT_DIR /
    "feature_integration" /
    "train_features.csv"
)

TEST_FEATURES_PATH = (
    OUTPUT_DIR /
    "feature_integration" /
    "test_features.csv"
)

print("Path configuration OK.")

Path configuration OK.


## Cek TRAIN dan TEST

In [18]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("TRAIN:", train_df.shape)
print("TEST :", test_df.shape)

assert len(train_df) == 64000
assert len(test_df) == 16000

assert "dev_status" in train_df.columns
assert "dev_status" in test_df.columns

print("\n✅ TRAIN = 64.000")
print("✅ TEST  = 16.000")
print("✅ Split 80/20 sesuai")

TRAIN: (64000, 8)
TEST : (16000, 8)

✅ TRAIN = 64.000
✅ TEST  = 16.000
✅ Split 80/20 sesuai


## Cek Target Tidak Masuk Feature

In [19]:
TARGET = "dev_status"

FEATURE_COLUMNS = [
    "weight",
    "height",
    "age",
    "gross_motor",
    "fine_motor_adaptive",
    "language",
    "personal_social"
]

assert TARGET not in FEATURE_COLUMNS

print("Feature:")
for feature in FEATURE_COLUMNS:
    print("-", feature)

print("\nTarget:", TARGET)

print("\n✅ dev_status tidak digunakan sebagai input feature.")

Feature:
- weight
- height
- age
- gross_motor
- fine_motor_adaptive
- language
- personal_social

Target: dev_status

✅ dev_status tidak digunakan sebagai input feature.


## Cek KG

In [20]:
# CELL 4 — Cek KG

kg_df = pd.read_csv(
    KG_PATH,
    sep="\t",
    header=0
)

expected_triples = (
    len(train_df) *
    len(FEATURE_COLUMNS)
)

actual_triples = len(kg_df)

print("Jumlah TRAIN:", len(train_df))
print("Jumlah feature KG:", len(FEATURE_COLUMNS))
print("Expected KG triples:", expected_triples)
print("Actual KG triples:", actual_triples)

assert set(kg_df.columns) == {
    "head",
    "relation",
    "tail"
}

target_relation_exists = (
    kg_df["relation"]
    .astype(str)
    .str.contains(
        "dev_status",
        case=False,
        na=False
    )
    .any()
)

assert not target_relation_exists

assert "has_dev_status" not in set(
    kg_df["relation"].astype(str)
)

print("\nUnique relations:")
print(
    sorted(
        kg_df["relation"]
        .astype(str)
        .unique()
    )
)

print(
    "\nTarget relation ditemukan:",
    target_relation_exists
)

print("\n✅ dev_status tidak masuk KG.")

Jumlah TRAIN: 64000
Jumlah feature KG: 7
Expected KG triples: 448000
Actual KG triples: 448000

Unique relations:
['has_age', 'has_fine_motor_adaptive', 'has_gross_motor', 'has_height', 'has_language', 'has_personal_social', 'has_weight']

Target relation ditemukan: False

✅ dev_status tidak masuk KG.


## Cek Relation KG

In [21]:
with open(
    KG_RELATIONS_PATH,
    "r",
    encoding="utf-8"
) as f:

    relations = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("Relations:")
for relation in relations:
    print("-", relation)

assert "has_dev_status" not in relations

print(
    "\n✅ has_dev_status tidak terdapat "
    "dalam relation KG."
)

Relations:
- has_age
- has_fine_motor_adaptive
- has_gross_motor
- has_height
- has_language
- has_personal_social
- has_weight

✅ has_dev_status tidak terdapat dalam relation KG.


## Cek Feature Integration

In [22]:
train_features = pd.read_csv(
    TRAIN_FEATURES_PATH
)

test_features = pd.read_csv(
    TEST_FEATURES_PATH
)

print(
    "Train features:",
    train_features.shape
)

print(
    "Test features :",
    test_features.shape
)

assert train_features.shape == (
    64000,
    107
)

assert test_features.shape == (
    16000,
    107
)

assert TARGET not in train_features.columns
assert TARGET not in test_features.columns

assert train_features.isna().sum().sum() == 0
assert test_features.isna().sum().sum() == 0

print("\n✅ Train feature = 64.000 × 107")
print("✅ Test feature  = 16.000 × 107")
print("✅ dev_status tidak masuk feature.")
print("✅ Tidak ada missing value.")

Train features: (64000, 107)
Test features : (16000, 107)

✅ Train feature = 64.000 × 107
✅ Test feature  = 16.000 × 107
✅ dev_status tidak masuk feature.
✅ Tidak ada missing value.


## Cek KGE Files

In [23]:
TRAIN_KGE_PATH = (
    OUTPUT_DIR /
    "kge_features" /
    "train_kge_embeddings.csv"
)

TEST_KGE_PATH = (
    OUTPUT_DIR /
    "kge_features" /
    "test_kge_embeddings.csv"
)

train_kge = pd.read_csv(
    TRAIN_KGE_PATH
)

test_kge = pd.read_csv(
    TEST_KGE_PATH
)

print(
    "Train KGE:",
    train_kge.shape
)

print(
    "Test KGE :",
    test_kge.shape
)

assert train_kge.shape == (
    64000,
    100
)

assert test_kge.shape == (
    16000,
    100
)

print("\n✅ KGE dimension = 100")
print("✅ TRAIN KGE = 64.000")
print("✅ TEST KGE  = 16.000")

Train KGE: (64000, 100)
Test KGE : (16000, 100)

✅ KGE dimension = 100
✅ TRAIN KGE = 64.000
✅ TEST KGE  = 16.000


## Cek Model Final

In [24]:
CATBOOST_MODEL = (
    MODEL_DIR /
    "catboost" /
    "catboost_final.cbm"
)

TRANSE_MODEL = (
    MODEL_DIR /
    "transe" /
    "transe_model_full_train.pkl"
)

assert CATBOOST_MODEL.exists()
assert TRANSE_MODEL.exists()

print(
    "CatBoost model:",
    CATBOOST_MODEL
)

print(
    "TransE model:",
    TRANSE_MODEL
)

print("\n✅ Model final tersedia.")

CatBoost model: ..\models\catboost\catboost_final.cbm
TransE model: ..\models\transe\transe_model_full_train.pkl

✅ Model final tersedia.


## Audit Struktur Notebook

In [25]:
NOTEBOOK_DIR = ROOT / "notebooks"

TRAINING_NOTEBOOKS = [
    "02_preprocessing.ipynb",
    "04_knowledge_graph.ipynb",
    "05_5_kge_representation.ipynb",
    "05_transe.ipynb",
    "06_feature_integration.ipynb",
    "07_catboostFold.ipynb",
    "08_evaluation.ipynb"
]

notebook_sources = {}

for notebook_name in TRAINING_NOTEBOOKS:

    path = NOTEBOOK_DIR / notebook_name

    if not path.exists():
        print(
            f"⚠️ Tidak ditemukan: {notebook_name}"
        )
        continue

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        notebook = json.load(f)

    source = "\n".join(
        "".join(
            cell.get("source", [])
        )
        for cell in notebook.get(
            "cells",
            []
        )
        if cell.get("cell_type") == "code"
    )

    notebook_sources[
        notebook_name
    ] = source

print(
    "\nNotebook berhasil diaudit:",
    len(notebook_sources)
)


Notebook berhasil diaudit: 7


## Cek Target Leakage pada Kode

In [26]:
# CELL 10 — Audit penggunaan target pada pipeline

# Kita tidak mencari sekadar teks "dev_status".
# Kita memeriksa apakah target benar-benar digunakan
# sebagai input fitur / KG / embedding.

target_leakage_found = []

for notebook_name, source in notebook_sources.items():

    # Hapus komentar agar pemeriksaan fokus pada kode aktif
    code_lines = []

    for line in source.splitlines():

        stripped = line.strip()

        if stripped.startswith("#"):
            continue

        code_lines.append(line)

    active_code = "\n".join(code_lines)

    # 1. dev_status tidak boleh menjadi bagian FEATURE_COLUMNS
    if re.search(
        r'FEATURE_COLUMNS\s*=.*dev_status',
        active_code,
        flags=re.IGNORECASE | re.DOTALL
    ):
        target_leakage_found.append(
            (
                notebook_name,
                "dev_status digunakan dalam FEATURE_COLUMNS"
            )
        )

    # 2. dev_status tidak boleh dimasukkan ke triples KG
    if re.search(
        r'has_dev_status',
        active_code,
        flags=re.IGNORECASE
    ):
        # Hanya dianggap masalah jika muncul sebagai
        # relation/entity yang benar-benar dibuat.
        if re.search(
            r'has_dev_status\s*[\'"]',
            active_code,
            flags=re.IGNORECASE
        ):
            target_leakage_found.append(
                (
                    notebook_name,
                    "has_dev_status digunakan sebagai relation/entity"
                )
            )

    # 3. dev_status tidak boleh dimasukkan ke embedding
    if re.search(
        r'(embedding|transe).*dev_status|dev_status.*(embedding|transe)',
        active_code,
        flags=re.IGNORECASE
    ):
        target_leakage_found.append(
            (
                notebook_name,
                "dev_status terhubung dengan proses embedding"
            )
        )


if target_leakage_found:

    print("❌ POTENSI TARGET LEAKAGE:")

    for notebook_name, reason in target_leakage_found:
        print(
            f"- {notebook_name}: {reason}"
        )

else:

    print(
        "✅ Tidak ditemukan penggunaan "
        "dev_status sebagai input pipeline."
    )

❌ POTENSI TARGET LEAKAGE:
- 02_preprocessing.ipynb: dev_status digunakan dalam FEATURE_COLUMNS
- 04_knowledge_graph.ipynb: dev_status digunakan dalam FEATURE_COLUMNS
- 04_knowledge_graph.ipynb: has_dev_status digunakan sebagai relation/entity
- 05_5_kge_representation.ipynb: dev_status digunakan dalam FEATURE_COLUMNS
- 05_5_kge_representation.ipynb: has_dev_status digunakan sebagai relation/entity
- 05_transe.ipynb: has_dev_status digunakan sebagai relation/entity
- 07_catboostFold.ipynb: dev_status digunakan dalam FEATURE_COLUMNS


## Final Audit Checklist

In [27]:
checks = {
    "TRAIN = 64000": len(train_df) == 64000,

    "TEST = 16000": len(test_df) == 16000,

    "Target tidak masuk feature":
        TARGET not in FEATURE_COLUMNS,

    "Target tidak masuk KG":
        "has_dev_status" not in relations,

    "Target tidak masuk train features":
        TARGET not in train_features.columns,

    "Target tidak masuk test features":
        TARGET not in test_features.columns,

    "Train KGE = 64000":
        len(train_kge) == 64000,

    "Test KGE = 16000":
        len(test_kge) == 16000,

    "Train features = 107":
        train_features.shape[1] == 107,

    "Test features = 107":
        test_features.shape[1] == 107,

    "CatBoost model tersedia":
        CATBOOST_MODEL.exists(),

    "TransE model tersedia":
        TRANSE_MODEL.exists(),

    "Tidak ditemukan target leakage pada kode":
        len(target_leakage_found) == 0
}


print("=" * 55)
print("CHECKPOINT 9 — DATA LEAKAGE AUDIT")
print("=" * 55)

for name, result in checks.items():

    status = "PASS" if result else "FAIL"

    print(
        f"[{status}] {name}"
    )

all_pass = all(
    checks.values()
)

print("\n" + "=" * 55)

if all_pass:

    print(
        "DATA LEAKAGE CHECK: PASS"
    )

else:

    print(
        "DATA LEAKAGE CHECK: FAILED"
    )

print("=" * 55)

CHECKPOINT 9 — DATA LEAKAGE AUDIT
[PASS] TRAIN = 64000
[PASS] TEST = 16000
[PASS] Target tidak masuk feature
[PASS] Target tidak masuk KG
[PASS] Target tidak masuk train features
[PASS] Target tidak masuk test features
[PASS] Train KGE = 64000
[PASS] Test KGE = 16000
[PASS] Train features = 107
[PASS] Test features = 107
[PASS] CatBoost model tersedia
[PASS] TransE model tersedia
[FAIL] Tidak ditemukan target leakage pada kode

DATA LEAKAGE CHECK: FAILED
